In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import pandas as pd
import numpy as np
import sys
from copy import deepcopy,copy
from datetime import datetime
import pickle
import sys
import matplotlib.pyplot as plt
import pynumdiff
import traceback
import time
# Import Machine Scientist
from importlib.machinery import SourceFileLoader
# Get the absolute path of the script's directory
script_dir = os.path.dirname('/export/home/oriolca/Integral_BMS_Governing_Equations/Lotka-Volterra/')
# Define the relative path to the module
relative_module_path = "../I-BMS-2d/parallel_ode.py"
path = os.path.join(script_dir, relative_module_path)
ms = SourceFileLoader("ms", path).load_module()

#Import prior
path = os.path.join(script_dir, "../I-BMS-2d/Prior/")
sys.path.append(path)
from fit_prior import read_prior_par
prior = read_prior_par('../I-BMS-2d/Prior/final_prior_param_sq.named_equations.nv2.np8.2016-09-09 18:49:42.800618.dat')

2025-10-30 11:59:21,048 [INFO] 
Limited Total Variation Regularization Support Detected! 
---> CVXPY is not installed. 
---> Many Total Variation Methods require CVXPY including: 
---> velocity, acceleration, jerk, jerk_sliding, smooth_acceleration
---> Please install CVXPY to use these methods.
---> Recommended to also install MOSEK and obtain a MOSEK license.
You can still use: total_variation_regularization.iterative_velocity

2025-10-30 11:59:21,049 [INFO] 
Limited Linear Model Support Detected! 
---> PYCHEBFUN is not installed. 
---> Install pychebfun to use chebfun derivatives (https://github.com/pychebfun/pychebfun/) 
You can still use other methods 

2025-10-30 11:59:21,049 [INFO] 
Limited Linear Model Support Detected! 
---> CVXPY is not installed. 
---> Install CVXPY to use lineardiff derivatives 
You can still use other methods 



In [2]:
data=pd.read_csv('noise_data/3.0_0.csv')

x={}
y={}
dx = {}

dy = {}

x['d0']=deepcopy(data)
y['d0']=deepcopy(data)
y['d0'].x=deepcopy(x['d0'].y)
y['d0'].y=deepcopy(x['d0'].x)

h = x['d0'].t.to_numpy()[1] - x['d0'].t.to_numpy()[0]

par = [2, 21, 21]
x_hat, dxdt_hat = pynumdiff.linear_model.polydiff(
        x['d0'].x, h, par, options=None)
y_hat, dydt_hat = pynumdiff.linear_model.polydiff(
        x['d0'].y, h, par, options=None)

dx['d0'] = [ pd.DataFrame(data={'x':x_hat,'y':y_hat}),
            pd.DataFrame(data={'x':dxdt_hat,'y':dydt_hat})]

dy['d0'] = [ pd.DataFrame(data={'x':y_hat,'y':x_hat}),
            pd.DataFrame(data={'x':dydt_hat,'y':dxdt_hat})]


mcmc_resets = 4
mcmc_steps = 5000
XLABS = ['x','y']
params = 8
print(x)
print(y)

print(dx)
print(dy)

description_lengths, mdl, mdl_x, mdl_y, mdl_model_x, mdl_model_y = (
        [],
        np.inf,
        np.inf,
        np.inf,
        None,
        None,
    )

del prior["Nopi_abs"]
del prior["Nopi2_abs"]
del prior["Nopi_sin"]
del prior["Nopi2_sin"]
del prior["Nopi_cos"]
del prior["Nopi2_cos"]
del prior["Nopi_tan"]
del prior["Nopi2_tan"]
del prior["Nopi_sinh"]
del prior["Nopi2_sinh"]
del prior["Nopi_cosh"]
del prior["Nopi2_cosh"]
del prior["Nopi_tanh"]
del prior["Nopi2_tanh"]
OPS = {
    #'sin': 1,
    #'cos': 1,
    #'tan': 1,
    "exp": 1,
    #'log': 1,
    #'sinh' : 1,
    #'cosh' : 1,
    #'tanh' : 1,
    "pow2": 1,
    "pow3": 1,
    #'sqrt' : 1,
    #'fac' : 1,
    "-": 1,
    "+": 2,
    "*": 2,
    "/": 2,
    "**": 2,
}

{'d0':      Unnamed: 0          x         y     t        dx        dy
0             0   7.328027  9.691377   0.0  2.325348 -3.233397
1             1  10.346662  7.696855   0.5  1.460501 -2.759917
2             2  13.266169  5.825056   1.0  0.743914 -2.336677
3             3  12.507730  5.641539   1.5  0.183323 -1.960165
4             4  12.882736  5.163144   2.0 -0.219504 -1.628592
..          ...        ...       ...   ...       ...       ...
155         155  14.067621  3.650732  77.5  1.032423 -0.095259
156         156  17.424126  4.496277  78.0  1.436035  0.009262
157         157  15.346661  2.172001  78.5  1.982093  0.093813
158         158  21.628174  0.915494  79.0  2.682108  0.187248
159         159  22.224839  2.724581  79.5  3.535731  0.328001

[160 rows x 6 columns]}
{'d0':      Unnamed: 0         x          y     t        dx        dy
0             0  9.691377   7.328027   0.0  2.325348 -3.233397
1             1  7.696855  10.346662   0.5  1.460501 -2.759917
2             2 

In [ ]:
dl, i_mdl, model_x, model_y = [], np.inf, None, None
Ts=[1] + [1.04**k for k in range(1, 40,2)]
pms_x = ms.Parallel(
    Ts,
    ops=OPS,
    variables=XLABS,
    parameters=["a%d" % i for i in range(params)],
    x=x,
    dx=dx,
    prior_par=prior,
)
pms_y = ms.Parallel(
    Ts,
    ops=OPS,
    variables=XLABS,
    parameters=["a%d" % i for i in range(params)],
    x=y,
    dx=dy,
    prior_par=prior,
)
print("setting f-g links")
for temp in pms_x.trees.keys():
    pms_x.trees[temp].fy = pms_y.trees[temp]
    pms_y.trees[temp].fy = pms_x.trees[temp]
    # print('refit')
    pms_x.trees[temp].get_bic(reset=True, fit=True)
    pms_x.trees[temp].get_energy(bic=True, reset=True)
pms_x.t1 = pms_x.trees[str(min(Ts))]
pms_y.t1 = pms_y.trees[str(min(Ts))]
print('Initial MCMC model x:',pms_x.t1,pms_x.t1.E)
print('Initial MCMC model y:',pms_y.t1,pms_y.t1.E)
mc_start = time.time()
description_lengths.append([])
stderr_fileno = sys.stderr
sys.stderr = open(os.devnull, 'w')
for i in range(1, mcmc_steps + 1):
    start = time.time()
    # MCMC update
    pms_x.mcmc_step(verbose = False)  # MCMC step within each T
    pms_y.mcmc_step(verbose = False)
    ET1, ET2 = (
        pms_x.tree_swap()
    )  # Attempt to swap two randomly selected consecutive temps

    if ET1 != None:
        t1 = pms_y.trees[ET1]
        t2 = pms_y.trees[ET2]
        BT1, BT2 = t1.BT, t2.BT
        pms_y.trees[ET1] = t2
        pms_y.trees[ET2] = t1
        t1.BT = BT2
        t2.BT = BT1
        pms_y.t1 = pms_y.trees[str(min(Ts))]
        # 

    dl.append(copy(pms_x.t1.E ))
    # Add the description length to the trace
    # description_lengths.append(pms.t1.E)
    # Check if this is the MDL expression so far
    if pms_x.t1.E < i_mdl:
        # if pms.t1.E==float('NaN'): print('NaN in best model mdl')
        i_mdl = copy(pms_x.t1.E)
        model_x = deepcopy(pms_x.t1)
        model_y = deepcopy(pms_y.t1)
    if i%10==0:
        print('Step',i,pms_x.t1,pms_x.t1.E,pms_y.t1, end='\r')

        b1= pms_x.t1.bic
        pms_x.t1.get_bic(reset=True,fit=True)
        b2 =pms_x.t1.bic

        if np.abs(b1-b2) > 1e-6:
            print('BIC X is wrong',b1,b2)

        E1= pms_x.t1.E
        pms_x.t1.get_energy(reset=True)
        E2 =pms_x.t1.E

        if np.abs(E1-E2) > 1e-6:
            print('BIC X is wrong',E1,E2)
        

setting f-g links
Initial MCMC model x: _a0_ 1034.79633637144
Initial MCMC model y: _a0_ 1034.79633637144
